In [1]:
import pandas as pd
import numpy as np

FILES = {
    "strict_mbu":             "data/raw/claims_strict_mbu.csv",
    "regular_sbu":             "data/raw/claims_regular_sbu.csv",
    "lenient_lbu":             "data/raw/claims_lenient_lbu.csv",
    "strict_lbu":              "data/raw/claims_strict_lbu.csv",
    "regular_mbu":             "data/raw/claims_regular_mbu.csv",
    "lenient_sbu":             "data/raw/claims_lenient_sbu.csv",
    "regular_mbu_highvolume":  "data/raw/claims_regular_mbu_highvolume.csv",
}

RAW = {}
for name, path in FILES.items():
    RAW[name] = pd.read_csv(path, dtype=str, keep_default_na=False, na_values=["", "NULL", "null"])
for name, df in RAW.items():
    print(f"{name:26} rows={len(df):6}  cols={df.shape[1]}")

strict_mbu                 rows=  1159  cols=47
regular_sbu                rows=   520  cols=47
lenient_lbu                rows=  2391  cols=47
strict_lbu                 rows=  2277  cols=47
regular_mbu                rows=  1075  cols=47
lenient_sbu                rows=   465  cols=47
regular_mbu_highvolume     rows= 25789  cols=47


In [2]:
cols = RAW["regular_mbu"].columns.tolist()
assert all(list(df.columns) == cols for df in RAW.values()), \
    "column mismatch between datasets"

print(f"{len(cols)} columns, identical across all seven:\n")
for i, c in enumerate(cols):
    print(f"{i:2}  {c}")

47 columns, identical across all seven:

 0  item_pk
 1  item_id
 2  report_pk
 3  employee_id
 4  report_id
 5  employee_id-2
 6  expense_type_name
 7  expense_category_name
 8  expense_category_code
 9  bill_number
10  bill_date
11  bill_amount
12  bill_currency_id
13  overridden_amount
14  mileage_unit
15  mileage_quantity
16  mileage_rate_per_unit
17  mileage_original_rate_per_unit
18  mileage_overridden_quantity
19  item_status
20  report_status
21  auto_approved
22  approval_date
23  rejection_date
24  policy_id
25  rule_id
26  limit_per_claim
27  limit_per_day
28  limit_per_month
29  limit_per_quarter
30  limit_per_year
31  allow_beyond_limit
32  frequency_per_day
33  frequency_per_month
34  frequency_per_quarter
35  frequency_per_year
36  allow_beyond_frequency
37  configured_reviewer_chain
38  configured_reviewer_count
39  remark_count
40  last_workflow_action
41  is_duplicate_flagged
42  duplicate_match_count
43  override_count
44  item_created_on
45  item_updated_on
46  subm

In [3]:
pd.set_option("display.max_rows", 100)
pd.DataFrame({
    "nulls":    df.isna().sum(),
    "null_%":   (df.isna().mean() * 100).round(1),
    "distinct": df.nunique(),
})

,nulls,null_%,distinct
item_pk,0,0.0,25789
item_id,0,0.0,25789
report_pk,0,0.0,19937
employee_id,0,0.0,75
report_id,0,0.0,19937
employee_id-2,0,0.0,75
expense_type_name,0,0.0,15
expense_category_name,0,0.0,6
expense_category_code,0,0.0,6
bill_number,7579,29.4,17814


In [6]:
DATE_COLS = [
    "bill_date", "submission_date", "approval_date", "rejection_date",
    "item_created_on", "item_updated_on",
]

NUM_COLS = [
    "bill_amount", "overridden_amount",
    "mileage_quantity", "mileage_rate_per_unit",
    "mileage_original_rate_per_unit", "mileage_overridden_quantity",
    "limit_per_claim", "limit_per_day", "limit_per_month",
    "limit_per_quarter", "limit_per_year",
    "frequency_per_day", "frequency_per_month",
    "frequency_per_quarter", "frequency_per_year",
    "duplicate_match_count", "override_count", "remark_count",
    "configured_reviewer_count",
]

BOOL_COLS = ["auto_approved", "is_duplicate_flagged",
             "allow_beyond_limit", "allow_beyond_frequency"]

ID_COLS = ["item_pk", "report_pk", "employee_id", "bill_currency_id"]


def cast_types(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    assert (df["employee_id"] == df["employee_id-2"]).all(), \
        "employee ids differ"
    df = df.drop(columns=["employee_id-2"])

    for c in DATE_COLS:
        df[c] = pd.to_datetime(df[c], errors="coerce")

    for c in NUM_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    for c in BOOL_COLS:
        df[c] = df[c].map({"TRUE": True, "True": True, "true": True,
                           "FALSE": False, "False": False, "false": False})

    for c in ID_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

    df["bill_number"] = df["bill_number"].str.strip()
    df["expense_category_code"] = df["expense_category_code"].astype("category")

    return df


TYPED = {name: cast_types(df) for name, df in RAW.items()}

print(TYPED["regular_mbu"].dtypes.value_counts())
TYPED["regular_mbu"][["item_id", "employee_id", "expense_category_code",
                       "bill_amount", "item_status", "bill_date",
                       "submission_date"]].head()

object            12
float64           12
int64              7
datetime64[ns]     6
Int64              4
bool               4
category           1
Name: count, dtype: int64


,item_id,employee_id,expense_category_code,bill_amount,item_status,bill_date,submission_date
0,TRVL-501,1000,TRVL,6136.25,Paid,2025-07-15,2025-07-29 00:00:00.000000000
1,MEAL-502,1000,MEAL,2954.83,Accepted,2025-01-30,2025-02-01 00:00:00.000000000
2,MEAL-503,1000,MEAL,3211.11,Accepted,2025-03-17,2025-03-24 23:45:02.071124091
3,MEAL-504,1000,MEAL,3369.99,Accepted,2025-04-29,2025-05-01 02:46:20.405393764
4,MEAL-505,1000,MEAL,3105.88,Paid,2025-06-23,2025-06-25 11:03:04.776940889


In [7]:
def add_derived(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # label--> approval
    df["is_approved"] = df["item_status"].isin(["Accepted", "Paid"])
    # label--> rejected
    df["is_rejected"] = df["item_status"] == "Rejected"

    # a CLEAN approval
    df["is_clean_approval"] = (
        df["is_approved"]
        & (df["auto_approved"] != True)
        & (df["override_count"].fillna(0) == 0)
    )

    # timing
    df["submit_lag_days"] = (df["submission_date"] - df["bill_date"]).dt.days
    df["bill_period"] = df["bill_date"].dt.to_period("M")

    # mileage claims
    df["is_mileage"] = df["mileage_quantity"].notna()

    return df


FINAL = {name: add_derived(df) for name, df in TYPED.items()}

for name, df in FINAL.items():
    print(f"{name:26} approved={df.is_approved.sum():5}  "
          f"rejected={df.is_rejected.sum():5}  "
          f"clean={df.is_clean_approval.sum():5}  "
          f"mileage={df.is_mileage.sum():4}")

strict_mbu                 approved=  877  rejected=  282  clean=  827  mileage= 138
regular_sbu                approved=  468  rejected=   52  clean=  450  mileage=  61
lenient_lbu                approved= 2359  rejected=   32  clean= 2306  mileage= 357
strict_lbu                 approved= 1715  rejected=  562  clean= 1634  mileage= 329
regular_mbu                approved=  966  rejected=  109  clean=  930  mileage= 215
lenient_sbu                approved=  460  rejected=    5  clean=  429  mileage=  53
regular_mbu_highvolume     approved=23305  rejected= 2484  clean=22099  mileage=4152


In [9]:
CATEGORY_NAMES = (
    pd.concat(FINAL.values())[["expense_category_code", "expense_category_name"]]
    .drop_duplicates()
    .set_index("expense_category_code")["expense_category_name"]
    .to_dict()
)
print(CATEGORY_NAMES)

def structural_checks(name, df):
    clean = df[df.is_clean_approval]

    mixed_reports = df.groupby("report_pk")["item_status"].nunique()

    pairs_type = clean.groupby(
        ["employee_id", "expense_category_code", "expense_type_name"],
        observed=True
    ).size()

    pairs_cat = clean.groupby(
        ["employee_id", "expense_category_code"],
        observed=True
    ).size()

    return dict(
        dataset=name,
        rows=len(df),
        employees=df.employee_id.nunique(),
        reject_rate=f"{(df.item_status == 'Rejected').mean():.1%}",
        auto_approved=int(df.auto_approved.sum()),
        clean_approvals=int(len(clean)),
        # multi_item_reports=f"{(df.groupby('report_pk').size() > 1).mean():.0%}",
        # mixed_outcome_reports=int((mixed_reports > 1).sum()),
        eligible_pairs_type=f"{(pairs_type >= 3).sum()}/{len(pairs_type)} "
                             f"({(pairs_type>=3).mean():.0%})",
        eligible_pairs_category=f"{(pairs_cat >= 3).sum()}/{len(pairs_cat)} "
                                 f"({(pairs_cat>=3).mean():.0%})",
        duplicate_flagged=int(df.is_duplicate_flagged.sum()),
        overridden=int((df.override_count > 0).sum()),
    )


summary = pd.DataFrame([structural_checks(name, df) for name, df in FINAL.items()])
summary

{'MED': 'Medical', 'MEAL': 'Meals', 'INT': 'Internet & Telecom', 'TRVL': 'Travel', 'FUEL': 'Fuel & Mileage', 'LEARN': 'Learning & Development'}


,dataset,rows,employees,reject_rate,auto_approved,clean_approvals,multi_item_reports,mixed_outcome_reports,eligible_pairs_type,eligible_pairs_category,duplicate_flagged,overridden
0,strict_mbu,1159,25,24.3%,31,827,10%,66,56/87 (64%),52/70 (74%),5,19
1,regular_sbu,520,10,10.0%,2,450,14%,34,28/39 (72%),21/30 (70%),2,16
2,lenient_lbu,2391,50,1.3%,32,2306,13%,158,124/192 (65%),113/160 (71%),17,21
3,strict_lbu,2277,50,24.7%,48,1634,12%,147,104/162 (64%),98/134 (73%),18,33
4,regular_mbu,1075,25,10.1%,12,930,9%,50,65/96 (68%),61/82 (74%),6,24
5,lenient_sbu,465,10,1.1%,28,429,13%,23,23/38 (61%),21/30 (70%),1,3
6,regular_mbu_highvolume,25789,75,9.6%,417,22099,25%,3111,494/641 (77%),344/393 (88%),177,789


In [10]:
def eligibility_by_category(name, df):
    clean = df[df.is_clean_approval]
    pairs = clean.groupby(
        ["employee_id", "expense_category_code", "expense_type_name"],
        observed=True
    ).size().reset_index(name="n_claims")

    out = (pairs.groupby("expense_category_code", observed=True)
                .agg(pairs=("n_claims", "size"),
                     eligible=("n_claims", lambda s: (s >= 3).sum()))
                .reset_index())
    out["expense_category_name"] = out["expense_category_code"].map(CATEGORY_NAMES)
    out["eligible_pct"] = (out["eligible"] / out["pairs"]).map("{:.0%}".format)
    out.insert(0, "dataset", name)
    return out[["dataset", "expense_category_code", "expense_category_name",
                "pairs", "eligible", "eligible_pct"]]

per_category = pd.concat(
    [eligibility_by_category(name, df) for name, df in FINAL.items()],
    ignore_index=True
)
per_category

,dataset,expense_category_code,expense_category_name,pairs,eligible,eligible_pct
0,strict_mbu,FUEL,Fuel & Mileage,11,7,64%
1,strict_mbu,INT,Internet & Telecom,8,6,75%
2,strict_mbu,LEARN,Learning & Development,20,7,35%
3,strict_mbu,MEAL,Meals,17,12,71%
4,strict_mbu,MED,Medical,11,8,73%
5,strict_mbu,TRVL,Travel,20,16,80%
6,regular_sbu,FUEL,Fuel & Mileage,5,3,60%
7,regular_sbu,INT,Internet & Telecom,5,4,80%
8,regular_sbu,LEARN,Learning & Development,9,5,56%
9,regular_sbu,MEAL,Meals,13,12,92%


In [12]:
combined = pd.concat(
    [df.assign(dataset=name) for name, df in FINAL.items()],
    ignore_index=True
)

print("combined rows:", len(combined))
print(combined.groupby("dataset")["employee_id"].apply(lambda s: s.min()).to_dict())

combined rows: 33676
{'lenient_lbu': 1000, 'lenient_sbu': 1000, 'regular_mbu': 1000, 'regular_mbu_highvolume': 1000, 'regular_sbu': 1000, 'strict_lbu': 1000, 'strict_mbu': 1000}


In [10]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    combined, minimal=True,
    title="Synthetic Claims — All Seven Datasets"
)
profile.to_file("reports/synthetic_all_seven_profile.html")

Summarize dataset:  39%|███████▋            | 22/57 [00:01<00:02, 15.18it/s, Describe variable: limit_per_year]
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:58: UserWarning: Discarding nonzero nanoseconds in conversion.
  "max": pd.Timestamp.to_pydatetime(series.max()),
Summarize dataset:  46%|███████▎        | 26/57 [00:02<00:01, 18.67it/s, Describe variable: allow_beyond_limit]C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\li

In [11]:
ProfileReport(FINAL["regular_mbu_highvolume"], minimal=True,
              title="regular_mbu_highvolume").to_file(
    "reports/regular_mbu_highvolume_profile.html"
)

Summarize dataset:  39%|███████▊            | 22/56 [00:01<00:01, 30.05it/s, Describe variable: limit_per_year]
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
 22%|███████████████▉                                                          | 11/51 [00:01<00:03, 10.47it/s]C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:58: UserWarning: Discarding nonzero nanoseconds in conversion.
  "max": pd.Timestamp.to_pydatetime(series.max()),
Summarize dataset:  55%|████████████▏         | 31/56 [00:01<00:01, 22.03it/s, Describe variable: remark_count]C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversio

In [19]:
from ydata_profiling import ProfileReport
ProfileReport(FINAL["regular_mbu"], minimal=True,
              title="regular_mbu").to_file(
    "reports/regular_mbu_profile.html"
)

 10%|███████▏                                                                   | 5/52 [00:00<00:01, 46.32it/s]C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:58: UserWarning: Discarding nonzero nanoseconds in conversion.
  "max": pd.Timestamp.to_pydatetime(series.max()),
Summarize dataset:  39%|███████▋            | 22/57 [00:00<00:00, 64.86it/s, Describe variable: limit_per_year]C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib

In [20]:
from ydata_profiling import ProfileReport
ProfileReport(FINAL["regular_sbu"], minimal=True,
              title="regular_sbu").to_file(
    "reports/regular_sbu_profile.html"
)

Summarize dataset:  37%|███████            | 21/57 [00:00<00:00, 76.25it/s, Describe variable: limit_per_claim]C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:58: UserWarning: Discarding nonzero nanoseconds in conversion.
  "max": pd.Timestamp.to_pydatetime(series.max()),
Summarize dataset:  40%|██████▊          | 23/57 [00:00<00:00, 76.25it/s, Describe variable: limit_per_quarter]C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib

In [21]:
from ydata_profiling import ProfileReport
ProfileReport(FINAL["strict_mbu"], minimal=True,
              title="strict_mbu").to_file(
    "reports/strict_mbu_profile.html"
)

Summarize dataset:  37%|██████▎          | 21/57 [00:00<00:00, 84.29it/s, Describe variable: limit_per_quarter]C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:58: UserWarning: Discarding nonzero nanoseconds in conversion.
  "max": pd.Timestamp.to_pydatetime(series.max()),
Summarize dataset:  39%|███████▋            | 22/57 [00:00<00:00, 84.29it/s, Describe variable: limit_per_year]C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib

In [22]:
from ydata_profiling import ProfileReport
ProfileReport(FINAL["strict_lbu"], minimal=True,
              title="strict_lbu").to_file(
    "reports/strict_lbu_profile.html"
)

Summarize dataset:  37%|██████▎          | 21/57 [00:00<00:00, 73.55it/s, Describe variable: limit_per_quarter]C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:58: UserWarning: Discarding nonzero nanoseconds in conversion.
  "max": pd.Timestamp.to_pydatetime(series.max()),
Summarize dataset:  44%|███████▍         | 25/57 [00:00<00:00, 62.67it/s, Describe variable: frequency_per_day]C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib

In [23]:
from ydata_profiling import ProfileReport
ProfileReport(FINAL["lenient_lbu"], minimal=True,
              title="lenient_lbu").to_file(
    "reports/lenient_lbu_profile.html"
)

Summarize dataset:  37%|██████▎          | 21/57 [00:00<00:00, 68.56it/s, Describe variable: limit_per_quarter]C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:58: UserWarning: Discarding nonzero nanoseconds in conversion.
  "max": pd.Timestamp.to_pydatetime(series.max()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),

C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:58: UserWarning: Discarding nonzero nanose

In [24]:
from ydata_profiling import ProfileReport
ProfileReport(FINAL["lenient_sbu"], minimal=True,
              title="lenient_sbu").to_file(
    "reports/lenient_sbu_profile.html"
)

Summarize dataset:  35%|██████▋            | 20/57 [00:00<00:00, 42.22it/s, Describe variable: limit_per_month]C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:58: UserWarning: Discarding nonzero nanoseconds in conversion.
  "max": pd.Timestamp.to_pydatetime(series.max()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:58: UserWarning: Discarding nonzero nanosec

In [25]:
import os
os.makedirs("data/interim", exist_ok=True)

for name, df in FINAL.items():
    path = f"data/interim/claims_{name}_typed.parquet"
    df.to_parquet(path, index=False)
    print(f"saved {path}  ({len(df)} rows)")

saved data/interim/claims_strict_mbu_typed.parquet  (1159 rows)
saved data/interim/claims_regular_sbu_typed.parquet  (520 rows)
saved data/interim/claims_lenient_lbu_typed.parquet  (2391 rows)
saved data/interim/claims_strict_lbu_typed.parquet  (2277 rows)
saved data/interim/claims_regular_mbu_typed.parquet  (1075 rows)
saved data/interim/claims_lenient_sbu_typed.parquet  (465 rows)
saved data/interim/claims_regular_mbu_highvolume_typed.parquet  (25789 rows)
